# ALOHA Robot — ACT Training & Evaluation (Colab)

Train/evaluate an ALOHA policy in the same practical Colab style as the quadruped workflow, including **live streamed training logs**.

## Model selected (found on Hugging Face)
- **`lerobot/act_aloha_sim_transfer_cube_human`**
- Task: **`AlohaTransferCube-v0`**
- Framework: **LeRobot + gym-aloha**

This notebook does 4 things:
1. Installs LeRobot with ALOHA sim support
2. Evaluates the pretrained HF model
3. Fine-tunes ACT with live log streaming
4. Evaluates the latest fine-tuned checkpoint


## 1) Runtime setup

Use **GPU runtime** in Colab (`Runtime -> Change runtime type -> GPU`).


In [ ]:
!nvidia-smi

In [ ]:
# Clone LeRobot at the commit used by the model card for compatibility
!rm -rf /content/lerobot
!git clone https://github.com/huggingface/lerobot.git /content/lerobot
%cd /content/lerobot
!git checkout 3c0a209

In [ ]:
# Install LeRobot + ALOHA simulation extras
!pip -q install -U pip
!pip -q install -e ".[aloha]"

# Optional experiment tracking
!pip -q install wandb

# Sanity check: package import must work before training
import os
os.environ['PYTHONPATH'] = '/content/lerobot:' + os.environ.get('PYTHONPATH', '')
import lerobot
print('LeRobot import OK:', lerobot.__file__)


In [ ]:
# Quick sanity: imports
import torch
import gymnasium as gym
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2) Evaluate pretrained ALOHA model from Hugging Face

We will use:
- `policy.path = lerobot/act_aloha_sim_transfer_cube_human`
- `env.task = AlohaTransferCube-v0`


In [ ]:
# (Optional) login if you hit hub rate limits/private assets
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Evaluate pretrained policy (quick pass)
!lerobot-eval \
  --policy.path=lerobot/act_aloha_sim_transfer_cube_human \
  --output_dir=outputs/eval/act_aloha_transfer_cube_hf \
  --env.type=aloha \
  --env.task=AlohaTransferCube-v0 \
  --eval.n_episodes=100 \
  --eval.batch_size=20 \
  --policy.device=cuda \
  --policy.use_amp=false


In [ ]:
# Inspect evaluation outputs
import json, pathlib
base = pathlib.Path('outputs/eval/act_aloha_transfer_cube_hf')
print('Eval dir exists:', base.exists())

for p in sorted(base.rglob('*')):
    if p.is_file() and p.suffix in ['.json', '.txt', '.md']:
        print('-', p)

eval_json = None
for cand in base.rglob('eval_info.json'):
    eval_json = cand
    break

if eval_json:
    print()
    print('Found eval_info.json:', eval_json)
    data = json.loads(eval_json.read_text())
    # Print a compact preview
    if isinstance(data, dict):
        for k in list(data.keys())[:12]:
            print(k, '=>', str(data[k])[:200])
else:
    print('No eval_info.json found. Check command logs above.')

## 3) Fine-tuning with live training logs (quadruped-style)

This cell runs training and streams logs line-by-line in real time, similar to the quadruped notebook feel.


In [ ]:
# Optional: keep tracking offline to avoid login prompts in Colab
!wandb offline


In [ ]:
# Live training stream (like quadruped)
import subprocess
import sys

train_cmd = [
    "lerobot-train",
    "--output_dir=outputs/train/act_aloha_transfer_ft",
    "--policy.type=act",
    "--dataset.repo_id=lerobot/aloha_sim_transfer_cube_human",
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--wandb.enable=false",
    "--steps=80000",
    "--eval_freq=10000",
    "--save_freq=10000",
    "--eval.n_episodes=50",
    "--eval.batch_size=25",
    "--policy.device=cuda",
    "--policy.use_amp=false",
]

print("Running:")
print(" ".join(train_cmd))

process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

ret = process.wait()
print(f"\nTraining finished with exit code: {ret}")
if ret != 0:
    raise RuntimeError("Training failed. Check logs above.")


In [ ]:
# Evaluate latest fine-tuned checkpoint automatically
import pathlib
import subprocess

ckpt_root = pathlib.Path("outputs/train/act_aloha_transfer_ft/checkpoints")
if not ckpt_root.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {ckpt_root}")

candidates = sorted(ckpt_root.glob("*/pretrained_model"))
if not candidates:
    raise FileNotFoundError("No pretrained_model checkpoint found under checkpoints/*/")

policy_path = str(candidates[-1])
print("Using checkpoint:", policy_path)

cmd = [
    "lerobot-eval",
    f"--policy.path={policy_path}",
    "--output_dir=outputs/eval/act_aloha_transfer_ft_latest",
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--eval.n_episodes=200",
    "--eval.batch_size=20",
    "--policy.device=cuda",
    "--policy.use_amp=false",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Fine-tuned eval complete.")


In [ ]:
# (Optional) Preview latest rendered eval video
from pathlib import Path
from IPython.display import Video, display

video_dir = Path("outputs/eval/act_aloha_transfer_ft_latest/videos")
videos = sorted(video_dir.glob("*.mp4")) if video_dir.exists() else []

if not videos:
    print("No videos found yet at", video_dir)
else:
    print("Showing:", videos[-1])
    display(Video(str(videos[-1]), embed=True, width=720))


## 4) Compare pretrained vs fine-tuned


In [ ]:
import json, pathlib, pandas as pd

def load_success(eval_dir):
    eval_dir = pathlib.Path(eval_dir)
    f = None
    for cand in eval_dir.rglob('eval_info.json'):
        f = cand
        break
    if f is None:
        return None
    data = json.loads(f.read_text())

    # Try common keys
    for k in ['success_rate', 'eval/success_rate', 'metrics/success_rate']:
        if isinstance(data, dict) and k in data:
            return float(data[k])

    # Fallback: infer from per-episode entries
    if isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, list) and len(v) and isinstance(v[0], dict):
                if 'success' in v[0]:
                    vals = [int(x.get('success', 0)) for x in v]
                    return sum(vals) / max(1, len(vals))
    return None

rows = [
    ['HF pretrained', load_success('outputs/eval/act_aloha_transfer_cube_hf')],
    ['Fine-tuned', load_success('outputs/eval/act_aloha_transfer_ft_080000')],
]

df = pd.DataFrame(rows, columns=['run', 'success_rate'])
display(df)

## 5) Notes

- If training is interrupted, rerun the live training cell with the same output dir and add `--resume=true`.
- For faster first pass, reduce `--steps` and `--eval.n_episodes`.
- The preview cell shows rendered eval videos when available.
